In [ ]:
"""
dofbot_shelf_pick.py

Requirements:
 - ultralytics (YOLOv8) installed
 - opencv-python
 - numpy
 - speechrecognition
 - pyaudio (or use other microphone backend)
 - optional: cv2.aruco (OpenCV-contrib provides this)
 - (Install with pip if needed: pip install ultralytics opencv-python numpy SpeechRecognition pyaudio)

Notes:
 - Change CAMERA_INDEX to your Dofbot camera device (0,1 or /dev/videoX)
 - Change MODEL_PATH to your YOLO model weights path
 - Fill DOFBOT control functions with your robot's SDK or serial commands
 - Fill SHELF_ZS with the Z (height) values of each shelf in the robot base frame (in mm)
 - If you already have calibration files (camera_intrinsics.npz, cam2robot.npy) the script will load them
"""

import cv2
import numpy as np
from ultralytics import YOLO
import speech_recognition as sr
import time
import os
import sys
import json

# -----------------------------
# CONFIGURATION (CHANGE THESE)
# -----------------------------
CAMERA_INDEX = 0                      # device index or video path
MODEL_PATH = "runs/train/toy_animals_full/weights/best.pt"
CALIB_FILE = "camera_intrinsics.npz"  # saved camera intrinsics
CAM2ROBOT_FILE = "cam2robot.npy"      # saved 4x4 transform matrix
CHECKERBOARD = (6, 9)                 # internal corners (cols, rows)
SQUARE_SIZE_MM = 25.0                 # size of one checker square in mm
SHELF_ZS = {                          # shelf name -> Z height (mm) in robot base coordinates
    "bottom": 120.0,
    "middle": 240.0,
    "top": 360.0
}
# Map common voice phrases to YOLO labels (adjust to your trained class names)
VOICE_TO_LABEL = {
    "zebra": "zebra",
    "tiger": "tiger",
    "elephant": "elephant",
    "lion": "lion",
    "cheetah": "cheetah",
    "giraffe": "giraffe"
}

# Robot reach / safety bounds (example values)
ROBOT_X_LIMITS = (50, 400)  # mm
ROBOT_Y_LIMITS = (-200, 200)
ROBOT_Z_LIMITS = (0, 450)

# -----------------------------
# LOAD YOLO MODEL
# -----------------------------
print("Loading YOLO model...")
model = YOLO(MODEL_PATH)
print("Model loaded.")

# -----------------------------
# CAMERA OPEN
# -----------------------------
cap = cv2.VideoCapture(CAMERA_INDEX)
if not cap.isOpened():
    print(f"ERROR: cannot open camera {CAMERA_INDEX}")
    sys.exit(1)
print("Camera opened.")

# -----------------------------
# --- Calibration helpers ---
# -----------------------------
def calibrate_intrinsics_and_save(save_path=CALIB_FILE, num_frames=15):
    """
    Interactive intrinsic calibration using a printed checkerboard.
    Press 'c' to capture a frame (when checkerboard detected).
    Press 'q' to finish and compute calibration.
    """
    objp = np.zeros((CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
    objp[:, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)
    objp *= SQUARE_SIZE_MM  # scale to mm

    obj_points = []
    img_points = []
    captured = 0
    print("Calibration: show checkerboard to camera. Press 'c' to capture frame (when corners found). Press 'q' to compute.")

    while True:
        ret, frame = cap.read()
        if not ret:
            continue

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        found, corners = cv2.findChessboardCorners(gray, CHECKERBOARD, None)
        disp = frame.copy()
        if found:
            cv2.drawChessboardCorners(disp, CHECKERBOARD, corners, found)

        cv2.putText(disp, f"Captured: {captured}/{num_frames}", (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)
        cv2.imshow("Intrinsic Calibration", disp)
        key = cv2.waitKey(1) & 0xFF

        if key == ord('c') and found:
            obj_points.append(objp.copy())
            # refine corners
            criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
            corners2 = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), criteria)
            img_points.append(corners2)
            captured += 1
            print(f"Captured {captured}")
            if captured >= num_frames:
                break
        if key == ord('q'):
            break

    cv2.destroyWindow("Intrinsic Calibration")
    if len(obj_points) < 3:
        raise RuntimeError("Not enough calibration frames captured.")

    h, w = gray.shape[:2]
    ret, K, dist, rvecs, tvecs = cv2.calibrateCamera(obj_points, img_points, (w,h), None, None)
    np.savez(save_path, K=K, dist=dist, rvecs=rvecs, tvecs=tvecs)
    print(f"Saved intrinsics to {save_path}")
    return K, dist

def load_intrinsics(path=CALIB_FILE):
    if not os.path.exists(path):
        return None, None
    data = np.load(path, allow_pickle=True)
    return data['K'], data['dist']

def detect_aruco_and_solve_cam2robot(aruco_id=0, marker_length_mm=50.0, debug=False):
    """
    Option A: Place an ArUco marker at a known robot frame pose,
    detect it and compute cam -> robot transform.

    This function returns the 4x4 transform from camera frame to robot base frame.
    You must set the marker at a known robot coordinate and type in that coordinate.
    """
    # Choose ArUco dictionary
    try:
        aruco_dict = cv2.aruco.Dictionary_get(cv2.aruco.DICT_4X4_50)
        aruco_params = cv2.aruco.DetectorParameters_create()
    except Exception as e:
        raise RuntimeError("ArUco requires opencv-contrib. Install opencv-contrib-python.") from e

    K, dist = load_intrinsics()
    if K is None:
        raise RuntimeError("Intrinsics required before ArUco-based cam2robot estimation. Run intrinsics first.")

    print("ArUco detection mode. Place marker in camera view and press 'c' to capture when visible.")
    while True:
        ret, frame = cap.read()
        if not ret:
            continue
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        corners, ids, rejected = cv2.aruco.detectMarkers(gray, aruco_dict, parameters=aruco_params)
        disp = frame.copy()
        if ids is not None:
            cv2.aruco.drawDetectedMarkers(disp, corners, ids)
        cv2.imshow("ArUco detect", disp)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('c') and ids is not None:
            # find the requested id
            idx = None
            for i, idv in enumerate(ids.flatten()):
                if int(idv) == aruco_id:
                    idx = i
                    break
            if idx is None:
                print(f"ArUco id {aruco_id} not found; detected ids: {ids.flatten()}")
                continue
            # estimate pose
            rvec, tvec, _ = cv2.aruco.estimatePoseSingleMarkers([corners[idx]], marker_length_mm, K, dist)
            rvec = rvec[0][0]; tvec = tvec[0][0]
            print("Detected ArUco pose (camera frame): rvec:", rvec, " tvec (mm):", tvec)
            # Now ask user for the true robot-base coordinates of the marker:
            print("Enter the known robot-base coordinates (X Y Z) in mm of the ARUCO marker center (example: 200 0 250):")
            s = input().strip()
            Xr, Yr, Zr = map(float, s.split())
            # build transform: marker pose in camera frame -> want camera->robot such that:
            # P_robot = T_cam_to_robot @ P_cam
            # we know marker origin in camera frame (tvec), and marker origin in robot frame (Xr,Yr,Zr)
            # Also include rotation: rvec -> R_cam_marker ; we want R_robot_cam
            R_cam_marker, _ = cv2.Rodrigues(rvec)
            T_marker_cam = np.eye(4)
            T_marker_cam[:3,:3] = R_cam_marker
            T_marker_cam[:3,3] = tvec  # marker position in camera frame

            # Marker in robot frame:
            T_marker_robot = np.eye(4)
            T_marker_robot[:3,3] = [Xr, Yr, Zr]

            # We want T_cam_to_robot such that T_marker_robot = T_cam_to_robot @ T_marker_cam
            # Thus T_cam_to_robot = T_marker_robot @ inv(T_marker_cam)
            T_cam_to_robot = T_marker_robot @ np.linalg.inv(T_marker_cam)
            print("Computed T_cam_to_robot:\n", T_cam_to_robot)
            np.save(CAM2ROBOT_FILE, T_cam_to_robot)
            print(f"Saved camera->robot transform to {CAM2ROBOT_FILE}")
            cv2.destroyWindow("ArUco detect")
            return T_cam_to_robot
        if key == ord('q'):
            cv2.destroyWindow("ArUco detect")
            raise RuntimeError("ArUco-based calibration aborted by user.")

def load_cam2robot(path=CAM2ROBOT_FILE):
    if not os.path.exists(path):
        return None
    return np.load(path)

# -----------------------------
# PIXEL -> ROBOT coordinate conversion
# -----------------------------
def pixel_to_camera_coords(x_pix, y_pix, depth_mm, K):
    """
    Convert pixel coordinates (x,y) and depth (mm) in camera frame to 3D camera coords (Xc, Yc, Zc).
    K is camera matrix.
    """
    fx = K[0,0]; fy = K[1,1]
    cx = K[0,2]; cy = K[1,2]
    Xc = (x_pix - cx) * depth_mm / fx
    Yc = (y_pix - cy) * depth_mm / fy
    Zc = depth_mm
    return np.array([Xc, Yc, Zc])

def camera_to_robot(P_cam_xyz, T_cam_to_robot):
    P_cam_h = np.array([P_cam_xyz[0], P_cam_xyz[1], P_cam_xyz[2], 1.0])
    P_robot_h = T_cam_to_robot @ P_cam_h
    return P_robot_h[:3]

# -----------------------------
# DEPTH ESTIMATION FOR SHELF (NO DEPTH CAMERA)
# -----------------------------
def estimate_depth_from_shelf_label(shelf_label):
    """
    Returns depth (Z in camera frame) corresponding to the shelf plane.
    Approach:
      - We have shelf heights in ROBOT FRAME (SHELF_ZS).
      - We transform a point on the shelf (Xr,Yr,Zr) into camera frame using inverse of T_cam_to_robot,
        then use its Zc as the depth estimate for points on that shelf.
    This allows a per-shelf constant depth in the camera frame.
    """
    if shelf_label not in SHELF_ZS:
        raise KeyError(f"Unknown shelf label: {shelf_label}")

    # Use center point in robot base coordinates (approx)
    Xr_center = 200.0  # approximate middle X in robot frame (mm) - adjust if needed
    Yr_center = 0.0
    Zr = SHELF_ZS[shelf_label]
    P_robot = np.array([Xr_center, Yr_center, Zr, 1.0])
    T = load_cam2robot()
    if T is None:
        raise RuntimeError("camera->robot transform needed to estimate shelf depth. Run ArUco-based calibration or set CAM2ROBOT_FILE.")
    T_inv = np.linalg.inv(T)
    P_cam = T_inv @ P_robot
    depth_mm = P_cam[2]
    return float(depth_mm)

# -----------------------------
# DOFBOT CONTROL PLACEHOLDERS
# -----------------------------
def dofbot_move_to_xyz(x_mm, y_mm, z_mm, gripper_open=True):
    """
    Placeholder: Replace with Dofbot API call or inverse-kinematics + motion commands.

    Example:
      angles = dofbot_ik_inverse(x_mm, y_mm, z_mm)
      dofbot.set_joint_angles(angles)
      dofbot.control_gripper(open=gripper_open)

    We'll just print for demo.
    """
    # safety checks
    if not (ROBOT_X_LIMITS[0] <= x_mm <= ROBOT_X_LIMITS[1] and ROBOT_Y_LIMITS[0] <= y_mm <= ROBOT_Y_LIMITS[1]):
        print(f"[SAFETY] Target ({x_mm:.1f},{y_mm:.1f}) outside XY limits; aborting move.")
        return False
    if not (ROBOT_Z_LIMITS[0] <= z_mm <= ROBOT_Z_LIMITS[1]):
        print(f"[SAFETY] Target Z={z_mm:.1f} outside limits; aborting move.")
        return False

    print(f"DOFBOT MOVE -> X:{x_mm:.1f} Y:{y_mm:.1f} Z:{z_mm:.1f} GripperOpen:{gripper_open}")
    # TODO: integrate real robot commands here.
    time.sleep(1.0)  # simulate motion time
    return True

# -----------------------------
# VOICE COMMAND HANDLING
# -----------------------------
def listen_for_command(timeout=5, phrase_time_limit=4):
    r = sr.Recognizer()
    with sr.Microphone() as source:
        print("Listening for toy name (speak now)...")
        r.adjust_for_ambient_noise(source, duration=0.5)
        try:
            audio = r.listen(source, timeout=timeout, phrase_time_limit=phrase_time_limit)
        except sr.WaitTimeoutError:
            print("No speech detected (timeout).")
            return None
    try:
        text = r.recognize_google(audio)
        print("Heard:", text)
        return text.lower()
    except sr.UnknownValueError:
        print("Could not understand audio.")
        return None
    except sr.RequestError as e:
        print("Speech recognition request failed:", e)
        return None

def parse_toy_and_shelf_from_text(text):
    """
    Rudimentary parser: look for toy name keywords and shelf words.
    Example phrases:
    - "Pick zebra from top shelf"
    - "grab the tiger on middle shelf"
    Returns (toy_label, shelf_label)
    """
    if not text:
        return None, None
    toy = None; shelf = None
    for k,v in VOICE_TO_LABEL.items():
        if k in text:
            toy = v
            break
    for s in SHELF_ZS.keys():
        if s in text:
            shelf = s
            break
    # If shelf not spoken, assume middle
    if shelf is None:
        shelf = "middle"
    return toy, shelf

# -----------------------------
# UTILITY: pick best detection for label
# -----------------------------
def find_best_detection_by_label(results, label_name):
    """
    Chooses the detection box whose class label matches and which is most centered / largest.
    Returns (cx, cy, bbox) in pixel coordinates.
    """
    candidates = []
    for box in results[0].boxes:
        cls = int(box.cls[0])
        name = model.names[cls]
        if name != label_name:
            continue
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        w = x2 - x1; h = y2 - y1
        area = w*h
        cx = x1 + w/2; cy = y1 + h/2
        candidates.append((area, cx, cy, (x1,y1,x2,y2)))
    if not candidates:
        return None
    # pick largest area (closest / easiest to pick)
    candidates.sort(key=lambda x: x[0], reverse=True)
    _, cx, cy, bbox = candidates[0]
    return int(cx), int(cy), bbox

# -----------------------------
# MAIN: load or calibrate, then loop for voice-driven picks
# -----------------------------
def main_loop():
    # Load intrinsics or run calibration
    K, dist = load_intrinsics()
    if K is None:
        print("No intrinsics found. Starting intrinsic calibration.")
        K, dist = calibrate_intrinsics_and_save()
    else:
        print("Loaded camera intrinsics.")

    # Load camera->robot transform or let user compute via ArUco
    T_cam_to_robot = load_cam2robot()
    if T_cam_to_robot is None:
        print("No camera->robot transform found. Running ArUco-based setup.")
        T_cam_to_robot = detect_aruco_and_solve_cam2robot()
    else:
        print(f"Loaded camera->robot transform from {CAM2ROBOT_FILE}")

    print("Ready. Say the toy name and (optionally) the shelf (top/middle/bottom). Say 'exit' to stop.")

    while True:
        # Listen for a command
        text = listen_for_command()
        if text is None:
            continue
        if "exit" in text or "stop" in text or "quit" in text:
            print("Exiting.")
            break

        toy_label, shelf_label = parse_toy_and_shelf_from_text(text)
        if toy_label is None:
            print("No known toy name detected in the command.")
            continue
        print(f"Command parsed -> Toy: {toy_label}  Shelf: {shelf_label}")

        # Capture a frame and run YOLO
        ret, frame = cap.read()
        if not ret:
            print("Failed to grab frame from camera.")
            continue

        # undistort
        frame_ud = cv2.undistort(frame, K, dist)

        # Run YOLO
        results = model(frame_ud)

        # find best detection matching requested toy
        found = find_best_detection_by_label(results, toy_label)
        if found is None:
            print(f"No detection found for {toy_label}. Try again or reposition camera.")
            # Show annotated frame for debugging
            annotated = results[0].plot()
            cv2.imshow("Detections", annotated)
            cv2.waitKey(1500)
            cv2.destroyWindow("Detections")
            continue

        cx, cy, bbox = found
        x1,y1,x2,y2 = bbox
        print(f"Detected {toy_label} at pixel ({cx},{cy}); bbox={bbox}")

        # Estimate depth using shelf plane mapping
        try:
            depth_mm = estimate_depth_from_shelf_label(shelf_label)
            print(f"Estimated depth for shelf '{shelf_label}' = {depth_mm:.1f} mm (camera frame)")
        except Exception as e:
            print("Error estimating shelf depth:", e)
            continue

        # Convert pixel -> camera coords -> robot coords
        P_cam = pixel_to_camera_coords(cx, cy, depth_mm, K)
        P_robot = camera_to_robot(P_cam, T_cam_to_robot)
        Xr, Yr, Zr = float(P_robot[0]), float(P_robot[1]), float(P_robot[2])
        print(f"Target in ROBOT frame (mm): X={Xr:.1f}, Y={Yr:.1f}, Z={Zr:.1f}")

        # Approach: move above the object first (safety)
        approach_Z = Zr + 80.0  # approach 8cm above shelf surface; tune as needed
        success = dofbot_move_to_xyz(Xr, Yr, approach_Z, gripper_open=True)
        if not success:
            print("Approach move aborted.")
            continue

        # Move down to grasp
        grasp_Z = Zr + 10.0  # lower close to object (10mm above shelf center). Tweak as needed
        success = dofbot_move_to_xyz(Xr, Yr, grasp_Z, gripper_open=True)
        if not success:
            print("Grasp move aborted.")
            continue

        # Close gripper (placeholder)
        print("Closing gripper...")
        # TODO: call gripper close command on your robot SDK
        time.sleep(0.8)

        # Lift up
        dofbot_move_to_xyz(Xr, Yr, approach_Z, gripper_open=False)

        print(f"Pick of {toy_label} complete. Place or return as needed.")

        # Show annotated detection briefly
        annotated = results[0].plot()
        cv2.rectangle(annotated, (x1,y1), (x2,y2), (0,255,0), 2)
        cv2.putText(annotated, f"{toy_label}", (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)
        cv2.imshow("Pick result", annotated)
        cv2.waitKey(1200)
        cv2.destroyWindow("Pick result")

    # Cleanup
    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    try:
        main_loop()
    except KeyboardInterrupt:
        print("Interrupted by user. Exiting.")
    except Exception as e:
        print("Fatal error:", e)
        raise
